# DS605 Lab 4: Feature Engineering for Airbnb Price Prediction

This notebook takes the cleaned dataset (`AB_NYC_2019_cleaned.csv`) and turns it into a model-ready feature set, based on what the analysis notebook found: `room_type` and `neighbourhood_group`/`neighbourhood` are the strongest price signals, while the numeric behavioural columns (reviews, availability, minimum nights) barely correlate with price.


## 1. Load the Cleaned Dataset

In [1]:
import numpy as np
import pandas as pd

# `df` stays exactly as loaded, untouched, for the rest of the notebook -- it's our
# reference copy of the cleaned data. All feature engineering happens on `df1`, a
# separate copy, so we can always compare back to the original if something looks off.
df = pd.read_csv('AB_NYC_2019_cleaned.csv')
df1 = df.copy()


In [2]:
df1.head()


,name,host_id,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,calculated_host_listings_count,availability_365,days_since_last_review
0,Clean & quiet apt home by the park,2787,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,6,365,262.0
1,Skylit Midtown Castle,2845,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2,355,48.0
2,THE VILLAGE OF HARLEM....NEW YORK !,4632,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,1,365,0.0
3,Cozy Entire Floor of Brownstone,4869,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,1,194,3.0
4,Entire Apt: Spacious Studio/Loft by central park,7192,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,1,0,231.0


## 2. Feature Engineering Plan

Based on the analysis notebook:

- **Keep and encode**: `room_type`, `neighbourhood_group`, `neighbourhood` -- these showed real, sizable price differences in the bivariate analysis.
- **Keep as-is (numeric)**: `minimum_nights`, `number_of_reviews`, `calculated_host_listings_count`, `availability_365`, `days_since_last_review`, `latitude`, `longitude` -- individually weak correlations with price, but cheap to include and tree-based models can still pick up minor, non-linear signal from them.
- **Transform**: `price` is heavily right-skewed, so a log-transformed target (`log_price`) will be used for training.
- **Reduce before encoding**: `neighbourhood` has 200+ unique values -- one-hot encoding all of them would add hundreds of mostly-empty columns, so rare neighbourhoods get grouped into `"Other"` first.
- **Drop (not usable as direct model inputs)**: `host_id` (an identifier, not a feature -- `calculated_host_listings_count` already summarizes host size numerically) and `name` (raw free text -- kept in `df1` for now, since it will be used separately for keyword-based features next, but excluded from the numeric feature matrix here).


## 3. Encode Low-Cardinality Categoricals: `room_type` and `neighbourhood_group`

In [3]:
# Both columns have only a handful of categories (3 room types, 5 boroughs), so a
# straightforward one-hot encoding works well here. drop_first=True avoids the "dummy
# variable trap" (perfect multicollinearity) for models like linear regression --
# tree-based models don't need this, but it doesn't hurt them either.
low_cardinality_cols = ['room_type', 'neighbourhood_group']

df1 = pd.get_dummies(df1, columns=low_cardinality_cols, drop_first=True, dtype=int)

df1.head()


,name,host_id,neighbourhood,latitude,longitude,price,minimum_nights,number_of_reviews,calculated_host_listings_count,availability_365,days_since_last_review,room_type_Private room,room_type_Shared room,neighbourhood_group_Brooklyn,neighbourhood_group_Manhattan,neighbourhood_group_Queens,neighbourhood_group_Staten Island
0,Clean & quiet apt home by the park,2787,Kensington,40.64749,-73.97237,149,1,9,6,365,262.0,1,0,1,0,0,0
1,Skylit Midtown Castle,2845,Midtown,40.75362,-73.98377,225,1,45,2,355,48.0,0,0,0,1,0,0
2,THE VILLAGE OF HARLEM....NEW YORK !,4632,Harlem,40.80902,-73.94190,150,3,0,1,365,0.0,1,0,0,1,0,0
3,Cozy Entire Floor of Brownstone,4869,Clinton Hill,40.68514,-73.95976,89,1,270,1,194,3.0,0,0,1,0,0,0
4,Entire Apt: Spacious Studio/Loft by central park,7192,East Harlem,40.79851,-73.94399,80,10,9,1,0,231.0,0,0,0,1,0,0


## 4. Reduce and Encode `neighbourhood` (High Cardinality)

In [4]:
# `neighbourhood` has far too many unique values to one-hot encode directly (it would add
# hundreds of columns, most of them almost always 0). Instead, keep only the most common
# neighbourhoods as their own category, and group everything else into "Other".
TOP_N_NEIGHBOURHOODS = 20   # matches the "15 most common neighbourhoods" scale used in the EDA, with a bit of headroom

top_neighbourhoods = df1['neighbourhood'].value_counts().head(TOP_N_NEIGHBOURHOODS).index

df1['neighbourhood_grouped'] = np.where(
    df1['neighbourhood'].isin(top_neighbourhoods),
    df1['neighbourhood'],
    'Other'
)

print(f"Reduced from {df1['neighbourhood'].nunique()} unique neighbourhoods to "
      f"{df1['neighbourhood_grouped'].nunique()} categories (top {TOP_N_NEIGHBOURHOODS} + 'Other').")

# One-hot encode the reduced-cardinality version, then drop the original full-detail column
# (it's now redundant -- neighbourhood_grouped carries the modelling-relevant information).
df1 = pd.get_dummies(df1, columns=['neighbourhood_grouped'], drop_first=True, dtype=int)
df1 = df1.drop(columns=['neighbourhood'])

df1.head()


Reduced from 221 unique neighbourhoods to 21 categories (top 20 + 'Other').


,name,host_id,latitude,longitude,price,minimum_nights,number_of_reviews,calculated_host_listings_count,availability_365,days_since_last_review,...,neighbourhood_grouped_Harlem,neighbourhood_grouped_Hell's Kitchen,neighbourhood_grouped_Lower East Side,neighbourhood_grouped_Midtown,neighbourhood_grouped_Other,neighbourhood_grouped_Upper East Side,neighbourhood_grouped_Upper West Side,neighbourhood_grouped_Washington Heights,neighbourhood_grouped_West Village,neighbourhood_grouped_Williamsburg
0,Clean & quiet apt home by the park,2787,40.64749,-73.97237,149,1,9,6,365,262.0,...,0,0,0,0,1,0,0,0,0,0
1,Skylit Midtown Castle,2845,40.75362,-73.98377,225,1,45,2,355,48.0,...,0,0,0,1,0,0,0,0,0,0
2,THE VILLAGE OF HARLEM....NEW YORK !,4632,40.80902,-73.94190,150,3,0,1,365,0.0,...,1,0,0,0,0,0,0,0,0,0
3,Cozy Entire Floor of Brownstone,4869,40.68514,-73.95976,89,1,270,1,194,3.0,...,0,0,0,0,0,0,0,0,0,0
4,Entire Apt: Spacious Studio/Loft by central park,7192,40.79851,-73.94399,80,10,9,1,0,231.0,...,0,0,0,0,0,0,0,0,0,0


## 5. Transform the Target: `price` → `log_price`

In [5]:
# price is heavily right-skewed (a long tail of very expensive listings), which tends to hurt
# regression models trained directly on it. log1p compresses that tail; np.expm1 reverses it
# later to turn predictions back into dollar amounts.
df1['log_price'] = np.log1p(df1['price'])

df1[['price', 'log_price']].describe()


,price,log_price
count,48895.000000,48895.000000
mean,152.720687,4.736885
std,240.154170,0.695344
min,0.000000,0.000000
25%,69.000000,4.248495
50%,106.000000,4.672829
75%,175.000000,5.170484
max,10000.000000,9.210440


## 6. Drop Columns That Aren't Usable as Direct Model Inputs

In [6]:
# host_id is an identifier, not a feature -- calculated_host_listings_count already captures
# the useful signal (how large a host's portfolio is) numerically.
df1 = df1.drop(columns=['host_id'])

# `name` (raw listing text) is deliberately kept in df1 for now -- it isn't a usable numeric/
# categorical feature as-is, but the next step derives keyword-based features from it. It will
# be excluded from the numeric feature matrix (X) whenever a model is actually trained.
print("Remaining columns:")
print(list(df1.columns))


Remaining columns:
['name', 'latitude', 'longitude', 'price', 'minimum_nights', 'number_of_reviews', 'calculated_host_listings_count', 'availability_365', 'days_since_last_review', 'room_type_Private room', 'room_type_Shared room', 'neighbourhood_group_Brooklyn', 'neighbourhood_group_Manhattan', 'neighbourhood_group_Queens', 'neighbourhood_group_Staten Island', 'neighbourhood_grouped_Bedford-Stuyvesant', 'neighbourhood_grouped_Bushwick', 'neighbourhood_grouped_Chelsea', 'neighbourhood_grouped_Clinton Hill', 'neighbourhood_grouped_Crown Heights', 'neighbourhood_grouped_East Harlem', 'neighbourhood_grouped_East Village', 'neighbourhood_grouped_Financial District', 'neighbourhood_grouped_Flatbush', 'neighbourhood_grouped_Greenpoint', 'neighbourhood_grouped_Harlem', "neighbourhood_grouped_Hell's Kitchen", 'neighbourhood_grouped_Lower East Side', 'neighbourhood_grouped_Midtown', 'neighbourhood_grouped_Other', 'neighbourhood_grouped_Upper East Side', 'neighbourhood_grouped_Upper West Side', 

In [7]:
# Final check: confirm the engineered dataset has no missing values and looks as expected.
df1.info()


<class 'pandas.DataFrame'>
RangeIndex: 48895 entries, 0 to 48894
Data columns (total 36 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   name                                      48895 non-null  str    
 1   latitude                                  48895 non-null  float64
 2   longitude                                 48895 non-null  float64
 3   price                                     48895 non-null  int64  
 4   minimum_nights                            48895 non-null  int64  
 5   number_of_reviews                         48895 non-null  int64  
 6   calculated_host_listings_count            48895 non-null  int64  
 7   availability_365                          48895 non-null  int64  
 8   days_since_last_review                    48895 non-null  float64
 9   room_type_Private room                    48895 non-null  int64  
 10  room_type_Shared room                     488

In [8]:
# duplicating the datafrmae incase the bigrams dont hold 
# much vlaue in the results of the prediction : 
df2 = df1.copy()

## 7. A Note on Scaling

The numeric columns here (e.g. `minimum_nights`, `availability_365`) are on very different scales, but they are **deliberately not scaled in this notebook**. Scaling (e.g. `StandardScaler`) should be *fit* only on the training split, not the full dataset -- fitting it here, before the train/test split happens, would leak information from the test set into training. Scaling belongs inside the model-training pipeline (Task 2), applied after the split, not here in feature engineering.


---

`df1` is now a fully encoded, model-ready feature set (aside from the `name`-based keyword features, which come next). Next: derive bigram/trigram keyword categories from the listing names and one-hot encode those too, using a fresh copy of the *original* `df` so none of this encoding work is affected.


## 8. Keyword Features from Listing Names (Bigrams)

You noticed in the analysis notebook that certain word pairs in listing names (e.g. the "Beekman Tower" cluster) lined up with real price differences. This section tests that idea as a model feature: find the most common 2-word phrases (bigrams) across all listing names, then turn "does this listing's name contain phrase X" into a binary column for each one.

This runs on a **fresh copy of the original, untouched `df`** (not `df1`), so none of the encoding work already done is affected until the new columns are merged in at the very end.


In [9]:
from sklearn.feature_extraction.text import CountVectorizer

# Fresh copy of the ORIGINAL, untouched dataset -- this experiment stays separate
# from df1 until the new columns are merged in at the end.
df2 = df.copy()

# Lowercase every name, filling missing names with an empty string (rather than
# dropping rows) -- we need to keep every row here so these features can be lined
# up with df1 afterward.
desc = df2['name'].fillna('').str.lower()

TOP_N_BIGRAMS = 15   # keeping it simple -- just the most common word pairs

vect = CountVectorizer(stop_words='english', ngram_range=(2, 2))
X = vect.fit_transform(desc)

# Find the TOP_N_BIGRAMS most frequent word pairs across all listing names.
bigram_counts = np.asarray(X.sum(axis=0)).ravel()
top_bigram_idx = bigram_counts.argsort()[::-1][:TOP_N_BIGRAMS]
feature_names = vect.get_feature_names_out()
top_bigrams = [feature_names[i] for i in top_bigram_idx]

print("Top bigrams used as features:")
print(top_bigrams)


Top bigrams used as features:
['private room', 'central park', 'east village', 'private bedroom', 'bedroom apartment', 'cozy room', 'upper east', 'times square', 'west village', 'new york', 'bedroom apt', 'park slope', 'upper west', 'cozy bedroom', 'spacious bedroom']


In [10]:
# One-hot encode: one binary column per top bigram -- 1 if that listing's name
# contains the phrase, 0 otherwise. This is the "categorize name based on the
# most common labels" step: every listing gets tagged against each top phrase.
bigram_features = pd.DataFrame(
    {
        f"name_bigram_{phrase.replace(' ', '_')}": (X[:, vect.vocabulary_[phrase]] > 0).toarray().ravel().astype(int)
        for phrase in top_bigrams
    },
    index=df2.index,
)

bigram_features.head()


,name_bigram_private_room,name_bigram_central_park,name_bigram_east_village,name_bigram_private_bedroom,name_bigram_bedroom_apartment,name_bigram_cozy_room,name_bigram_upper_east,name_bigram_times_square,name_bigram_west_village,name_bigram_new_york,name_bigram_bedroom_apt,name_bigram_park_slope,name_bigram_upper_west,name_bigram_cozy_bedroom,name_bigram_spacious_bedroom
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0


In [11]:
# Merge the new bigram flag columns into df1, the main engineered feature set.
# Safe to align on index -- df1 and df2 both trace back to the same original df,
# and no rows have been dropped along the way.
df1 = pd.concat([df1, bigram_features], axis=1)

df1.tail()


,name,latitude,longitude,price,minimum_nights,number_of_reviews,calculated_host_listings_count,availability_365,days_since_last_review,room_type_Private room,...,name_bigram_cozy_room,name_bigram_upper_east,name_bigram_times_square,name_bigram_west_village,name_bigram_new_york,name_bigram_bedroom_apt,name_bigram_park_slope,name_bigram_upper_west,name_bigram_cozy_bedroom,name_bigram_spacious_bedroom
48890,Charming one bedroom - newly renovated rowhouse,40.67853,-73.94995,70,2,0,2,9,0.0,1,...,0,0,0,0,0,0,0,0,0,0
48891,Affordable room in Bushwick/East Williamsburg,40.70184,-73.93317,40,4,0,2,36,0.0,1,...,0,0,0,0,0,0,0,0,0,0
48892,Sunny Studio at Historical Neighborhood,40.81475,-73.94867,115,10,0,1,27,0.0,0,...,0,0,0,0,0,0,0,0,0,0
48893,43rd St. Time Square-cozy single bed,40.75751,-73.99112,55,1,0,6,2,0.0,0,...,0,0,0,0,0,0,0,0,0,0
48894,Trendy duplex in the very heart of Hell's Kitchen,40.76404,-73.98933,90,7,0,1,23,0.0,1,...,0,0,0,0,0,0,0,0,0,0


**Worth knowing before you train on this:** the bigrams above are selected purely by *frequency*, not by price -- so the top 15 will likely be generic phrases like `private room`, `central park`, or `east village`, not the specific price-diagnostic phrases (e.g. `beekman tower`, `fee beekman`) your analysis actually found to matter. Those were high-*price* but only moderately frequent (30-50 listings), so a pure top-frequency cutoff can miss them. If you want the model to actually capture that signal, the next step could be adding a small, explicit set of price-relevant phrases (from your analysis notebook's findings) alongside these frequency-based ones, rather than relying on frequency alone.


In [ ]:
# dropping the name column : 
df1 = df1.drop(columns=['name'])

In [13]:
df1.to_csv("df1.csv", index=False)

In [15]:
df2.to_csv("df2.csv", index=False)